# Assignment 3 – Regularization & Optimization

---

| Field | Details |
|---|---|
| **Assignment** | Assignment 3 – Unit 2: Regularization & Optimization |
| **Student Name** | Sachin Shrestha |
| **Student ID** | 032338-22 |
| **Date** | 18th April, 2026 |
| **Dataset** | EuroSAT (Land Use and Land Cover Classification) |


---
## 1. Objective

- Understand how **L1, L2, and Dropout** regularization combat overfitting in neural networks.
- Compare the convergence behavior and final performance of **SGD** vs **Adam** optimizers.
- Investigate the effect of **Batch Normalization** on training stability and generalization.
- Visualize **train vs. validation loss curves** to diagnose overfitting and underfitting.
- Identify the best combination of regularization + optimizer for the **EuroSAT** satellite image land-use classification task.


---
## 2. Theoretical Background

### 2.1 Regularization

Regularization techniques reduce model complexity and prevent overfitting by penalizing large weights or randomly deactivating neurons.

#### L2 Regularization (Weight Decay)
Adds the squared magnitude of all weights to the loss function:
$$\mathcal{L}_{total} = \mathcal{L}_{CE} + \lambda \sum_i w_i^2$$
This encourages weights to be small but non-zero, distributing information across many weights.

#### L1 Regularization
Adds the absolute magnitude of weights to the loss:
$$\mathcal{L}_{total} = \mathcal{L}_{CE} + \lambda \sum_i |w_i|$$
Promotes **sparsity** — many weights are driven to exactly zero, enabling feature selection.

#### Dropout
During training, each neuron is independently set to zero with probability $p$:
$$\tilde{h} = h \cdot \text{Bernoulli}(1-p)$$
This acts as ensemble training over $2^n$ sub-networks, reducing co-adaptation of neurons.

#### Batch Normalization
Normalizes layer inputs across the mini-batch:
$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \quad y_i = \gamma \hat{x}_i + \beta$$
Stabilizes gradients, allows higher learning rates, and provides mild regularization.

### 2.2 Optimizers

#### SGD with Momentum
$$v_t = \mu v_{t-1} - \eta \nabla \mathcal{L}$$
$$w_t = w_{t-1} + v_t$$
Simple and reliable, but sensitive to the learning rate. Often generalized better with careful tuning.

#### Adam (Adaptive Moment Estimation)
Maintains running averages of gradients ($m_t$) and squared gradients ($v_t$):
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t, \quad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$$
$$w_t = w_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$
Adapts learning rates per-parameter; converges faster but may generalize slightly worse than well-tuned SGD.

---
## 3. Dataset Description

| Property | Value |
|---|---|
| **Dataset** | EuroSAT – Land Use and Land Cover Classification |
| **Source** | `torchvision.datasets.EuroSAT` |
| **Classes** | 10 land-use categories (AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial, Pasture, PermanentCrop, Residential, River, SeaLake) |
| **Total Samples** | 27,000 RGB images (subsampled to 200/30/30 per class for training) |
| **Image Size** | 32 × 32 pixels (resized for fast experimentation) |
| **Split Strategy** | 200 train / 30 val / 30 test per class (stratified, seed=42) |
| **Preprocessing** | Resize to 32×32; normalize with EuroSAT mean & std; random rotation + color jitter for training |

> **Why EuroSAT?** The dataset contains high-resolution satellite images across 10 spectrally similar land-use classes (e.g., AnnualCrop vs PermanentCrop look visually similar). This subtle inter-class similarity makes overfitting likely on a plain CNN without regularization, making the with/without regularization contrast clear and educationally informative.


---
## 4. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')


---
## 5. Dataset Loading & Preprocessing

In [ ]:
# EuroSAT statistics (computed from the dataset)
EUROSAT_MEAN = (0.3444, 0.3803, 0.4078)
EUROSAT_STD  = (0.2034, 0.1366, 0.1148)

# Training transform: resize to 32x32 + augmentation + normalization
train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(EUROSAT_MEAN, EUROSAT_STD),
])

# Validation / test transform: resize + normalization only
test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(EUROSAT_MEAN, EUROSAT_STD),
])

# Download the EuroSAT dataset
full_dataset = torchvision.datasets.EuroSAT(root='./data', download=True, transform=train_transform)

_samples = getattr(full_dataset, 'samples', None) or full_dataset.imgs

# ── Stratified subsample: 200 train / 30 val / 30 test per class (10 classes) 
TRAIN_PER_CLASS = 200
VAL_PER_CLASS   = 30
TEST_PER_CLASS  = 30

from collections import defaultdict
class_indices = defaultdict(list)
for idx, (_, label) in enumerate(_samples):
    class_indices[label].append(idx)

train_idx, val_idx, test_idx = [], [], []
rng = np.random.default_rng(42)
for label in sorted(class_indices):
    idxs = rng.permutation(class_indices[label]).tolist()
    train_idx += idxs[:TRAIN_PER_CLASS]
    val_idx   += idxs[TRAIN_PER_CLASS:TRAIN_PER_CLASS + VAL_PER_CLASS]
    test_idx  += idxs[TRAIN_PER_CLASS + VAL_PER_CLASS:TRAIN_PER_CLASS + VAL_PER_CLASS + TEST_PER_CLASS]

train_set = torch.utils.data.Subset(full_dataset, train_idx)

# Apply test transform to val and test subsets via a wrapper
class TransformSubset(torch.utils.data.Dataset):
    #Wraps a Subset and applies a different transform.
    def __init__(self, base_dataset, indices, transform):
        self.dataset   = base_dataset
        self.indices   = indices
        self.transform = transform
        self._samples  = getattr(base_dataset, 'samples', None) or base_dataset.imgs
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        from PIL import Image as PILImage
        path, label = self._samples[self.indices[idx]]
        pil = PILImage.open(path).convert('RGB')
        return self.transform(pil), label

val_set_clean  = TransformSubset(full_dataset, val_idx,  test_transform)
test_set_clean = TransformSubset(full_dataset, test_idx, test_transform)

# Batch size = 128 as required
BATCH_SIZE = 128
PIN_MEMORY = torch.cuda.is_available()
train_loader = torch.utils.data.DataLoader(train_set,      batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=PIN_MEMORY)
val_loader   = torch.utils.data.DataLoader(val_set_clean,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)
test_loader  = torch.utils.data.DataLoader(test_set_clean, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)

# EuroSAT class names (10 classes)
CLASSES = (
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway',
    'Industrial', 'Pasture', 'PermanentCrop', 'Residential',
    'River', 'SeaLake'
)

print(f'Batch size    : {BATCH_SIZE}')
print(f'Train samples : {len(train_set):,}  ({len(train_loader)} batches)')
print(f'Val   samples : {len(val_set_clean):,}  ({len(val_loader)} batches)')
print(f'Test  samples : {len(test_set_clean):,}  ({len(test_loader)} batches)')
print(f'Number of classes: {len(CLASSES)}')


In [ ]:
# Visualise a sample batch 
def imshow(img_tensor, title=None):
    """Un-normalize and display a tensor image."""
    mean = torch.tensor(EUROSAT_MEAN).view(3, 1, 1)
    std  = torch.tensor(EUROSAT_STD ).view(3, 1, 1)
    img  = img_tensor * std + mean          # un-normalize
    img  = img.permute(1, 2, 0).numpy()
    img  = np.clip(img, 0, 1)
    plt.imshow(img)
    if title:
        plt.title(title, fontsize=7)
    plt.axis('off')

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flatten()):
    plt.sca(ax)
    imshow(images[i], CLASSES[labels[i]])
plt.suptitle('Sample EuroSAT Training Images', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---
## 6. Model Architectures

We define a modular CNN that accepts flags to enable/disable **BatchNorm** and **Dropout**.

In [ ]:
class ConvNet(nn.Module):
    #Configurable CNN for EuroSAT (10-class land-use classification from 32x32 satellite images).
    def __init__(self, use_batchnorm: bool = False, dropout_p: float = 0.0, num_classes: int = 10):
        super().__init__()
        self.use_batchnorm = use_batchnorm
        self.dropout_p     = dropout_p

        # Feature extractor 
        def conv_block(in_ch, out_ch, kernel=3, pad=1):
            layers = [nn.Conv2d(in_ch, out_ch, kernel, padding=pad)]
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_ch))
            layers.append(nn.ReLU(inplace=True))
            return nn.Sequential(*layers)

        self.features = nn.Sequential(
            conv_block(3,  64),
            conv_block(64, 64),
            nn.MaxPool2d(2, 2),          # 32→16

            conv_block(64,  128),
            conv_block(128, 128),
            nn.MaxPool2d(2, 2),          # 16→8

            conv_block(128, 256),
            conv_block(256, 256),
            nn.MaxPool2d(2, 2),          # 8→4
        )

        # Classifier 
        # EuroSAT images are 32x32; after 3 MaxPool(2,2) → 4x4 feature maps
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(256, num_classes),
        )

        # Weight initialisation
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# Sanity-check with EuroSAT input size (32x32)
dummy      = torch.randn(4, 3, 32, 32)
model_test = ConvNet(use_batchnorm=True, dropout_p=0.5, num_classes=10)
out        = model_test(dummy)
print(f'Output shape: {out.shape}')  # Expected: (4, 10)
total_params = sum(p.numel() for p in model_test.parameters())
print(f'Total parameters: {total_params:,}')


---
## 7. Training Infrastructure

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, l1_lambda=0.0):
    """Run one training epoch; returns (avg_loss, accuracy).
    Note: L2 regularization (weight decay) is handled directly by the
    optimizer via the weight_decay parameter — no manual penalty needed.
    L1 regularization is applied manually here when l1_lambda > 0.
    """
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)

        # Manual L1 penalty on weights only (L2 handled via optimizer weight_decay)
        if l1_lambda > 0:
            l1_penalty = sum(p.abs().sum() for n, p in model.named_parameters() if 'weight' in n)
            loss = loss + l1_lambda * l1_penalty

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted  = outputs.max(1)
        correct      += predicted.eq(labels).sum().item()
        total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluate on loader; returns (avg_loss, accuracy)."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss    = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted  = outputs.max(1)
        correct      += predicted.eq(labels).sum().item()
        total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def run_experiment(config: dict, epochs: int = 10, verbose: bool = True):
    #Full training loop for a single experimental configuration.
    model = ConvNet(
        use_batchnorm=config.get('use_batchnorm', False),
        dropout_p    =config.get('dropout_p',     0.0),
        num_classes  =10,
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    opt_name = config.get('optimizer', 'adam').lower()
    lr       = config.get('lr', 1e-3)
    wd       = config.get('weight_decay', 0.0)

    if opt_name == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    l1_lambda = config.get('l1_lambda', 0.0)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, l1_lambda=l1_lambda)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'  ].append(vl_loss)
        history['train_acc' ].append(tr_acc)
        history['val_acc'   ].append(vl_acc)

        if verbose:
            print(f"  Epoch {epoch:02d}/{epochs} | "
                  f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.1f}% | "
                  f"Val   Loss: {vl_loss:.4f} Acc: {vl_acc:.1f}%")

    # Final test accuracy
    _, test_acc = evaluate(model, test_loader, criterion)
    history['test_acc'] = test_acc
    history['model']    = model
    print(f"  → [{config['name']}] Test Accuracy: {test_acc:.2f}%\n")
    return history


print('Training infrastructure ready.')


---
## 8. Experiments

We run **6 configurations** covering:
1. Baseline (no regularization, Adam)
2. L2 Regularization (Adam)
3. L1 Regularization (Adam)
4. Dropout (Adam)
5. Dropout + BatchNorm (Adam)
6. Dropout + BatchNorm (SGD) ← Optimizer Comparison



In [ ]:
EPOCHS = 10

EXPERIMENTS = [
    # Regularization comparison (all Adam) 
    {
        'name'         : 'Baseline (No Reg, Adam)',
        'optimizer'    : 'adam',
        'lr'           : 1e-3,
        'weight_decay' : 0.0,
        'l1_lambda'    : 0.0,
        'dropout_p'    : 0.0,
        'use_batchnorm': False,
    },
    {
        'name'         : 'L2 Reg (Adam, wd=1e-4)',
        'optimizer'    : 'adam',
        'lr'           : 1e-3,
        'weight_decay' : 1e-4,
        'l1_lambda'    : 0.0,
        'dropout_p'    : 0.0,
        'use_batchnorm': False,
    },
    {
        'name'         : 'L1 Reg (Adam, λ=1e-5)',
        'optimizer'    : 'adam',
        'lr'           : 1e-3,
        'weight_decay' : 0.0,
        'l1_lambda'    : 1e-5,
        'dropout_p'    : 0.0,
        'use_batchnorm': False,
    },
    {
        'name'         : 'Dropout p=0.4 (Adam)',
        'optimizer'    : 'adam',
        'lr'           : 1e-3,
        'weight_decay' : 0.0,
        'l1_lambda'    : 0.0,
        'dropout_p'    : 0.4,
        'use_batchnorm': False,
    },
    # Optimizer comparison (Dropout + BatchNorm) 
    {
        'name'         : 'Dropout+BN (Adam)',
        'optimizer'    : 'adam',
        'lr'           : 1e-3,
        'weight_decay' : 1e-4,
        'l1_lambda'    : 0.0,
        'dropout_p'    : 0.4,
        'use_batchnorm': True,
    },
    {
        'name'         : 'Dropout+BN (SGD)',
        'optimizer'    : 'sgd',
        'lr'           : 0.05,
        'weight_decay' : 1e-4,
        'l1_lambda'    : 0.0,
        'dropout_p'    : 0.4,
        'use_batchnorm': True,
    },
]

results = {}

for cfg in EXPERIMENTS:
    print(f"{'='*60}")
    print(f"Running: {cfg['name']}")
    print(f"{'='*60}")
    results[cfg['name']] = run_experiment(cfg, epochs=EPOCHS)


---
## 9. Results

### 9.1 Train vs Validation Loss – Regularization Comparison

In [ ]:
# ── Plot helper ──────────────────────────────────────────────────────────────
PALETTE = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c']

def plot_loss_curves(experiment_names, results_dict, title, figsize=(14, 5)):
    
    n = len(experiment_names)
    ncols = min(n, 2)
    nrows = (n + 1) // 2
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    # Flatten axes array for uniform indexing
    axes = np.array(axes).flatten() if n > 1 else [axes]

    for i, name in enumerate(experiment_names):
        h   = results_dict[name]
        col = PALETTE[i % len(PALETTE)]
        ep  = range(1, len(h['train_loss']) + 1)
        ax  = axes[i]
        ax.plot(ep, h['train_loss'], color=col,  lw=2.0, linestyle='-',  label='Train Loss')
        ax.plot(ep, h['val_loss'],   color=col,  lw=2.0, linestyle='--', label='Val Loss')
        ax.fill_between(ep, h['train_loss'], h['val_loss'], alpha=0.10, color=col)
        ax.set_title(name, fontsize=10, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=10)
        ax.set_ylabel('Loss', fontsize=10)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    # Hide any unused subplots
    for j in range(len(experiment_names), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(title, fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


# Regularization comparison (first 4 experiments) ── train vs val loss overlaid
reg_names = [cfg['name'] for cfg in EXPERIMENTS[:4]]
plot_loss_curves(reg_names, results,
                 'Regularization Comparison – Train vs Validation Loss (solid=train, dashed=val)',
                 figsize=(16, 8))


### 9.2 Train vs Validation Loss – Optimizer Comparison

In [ ]:
# SGD vs Adam (last 2 experiments) ── train vs val loss overlaid
opt_names = [cfg['name'] for cfg in EXPERIMENTS[-2:]]
plot_loss_curves(opt_names, results,
                 'Optimizer Comparison – Adam vs SGD (solid=train, dashed=val)',
                 figsize=(14, 5))


### 9.3 Overfitting Demonstration

In [ ]:
# Overfitting demonstration: Baseline vs best regularized model
baseline = results['Baseline (No Reg, Adam)']
best     = results['Dropout+BN (Adam)']

ep  = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

configs = [
    (baseline, 'Baseline – Severe Overfitting',          '#e74c3c', '#f1948a'),
    (best,     'Dropout + BatchNorm – Reduced Overfitting', '#2ecc71', '#82e0aa'),
]

for ax, (hist, title, col_tr, col_vl) in zip(axes, configs):
    # Compute train-val accuracy gap per epoch
    gap = [tr - vl for tr, vl in zip(hist['train_acc'], hist['val_acc'])]
    avg_gap = sum(gap) / len(gap)

    ax.plot(ep, hist['train_acc'], color=col_tr, lw=2, label='Train Acc')
    ax.plot(ep, hist['val_acc'],   color=col_vl, lw=2, linestyle='--', label='Val Acc')
    ax.fill_between(ep, hist['val_acc'], hist['train_acc'],
                    alpha=0.15, color='grey', label=f'Overfit Gap (avg {avg_gap:.1f}%)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Accuracy (%)', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(0, 100)

plt.suptitle('Overfitting Demonstration: Baseline vs Regularized Model',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### 9.4 Summary Table

In [ ]:
# Summary table: final metrics for all experiments
header = f"{'Experiment':<38} {'Train Acc':>10} {'Val Acc':>9} {'Test Acc':>10} {'Overfit Gap':>13}"
print(header)
print('-' * len(header))

for name, hist in results.items():
    tr_acc = hist['train_acc'][-1]
    vl_acc = hist['val_acc'][-1]
    te_acc = hist['test_acc']
    gap    = tr_acc - vl_acc
    print(f"{name:<38} {tr_acc:>9.2f}% {vl_acc:>8.2f}% {te_acc:>9.2f}% {gap:>12.2f}%")


### 9.5 Validation Accuracy Bar Chart

In [ ]:
names    = list(results.keys())
test_acc = [results[n]['test_acc'] for n in names]
overfit  = [results[n]['train_acc'][-1] - results[n]['val_acc'][-1] for n in names]

x    = np.arange(len(names))
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

bars1 = axes[0].barh(x, test_acc, color=PALETTE[:len(names)], edgecolor='white', height=0.55)
axes[0].set_yticks(x)
axes[0].set_yticklabels([n.replace(' (', '\n(') for n in names], fontsize=9)
axes[0].set_xlabel('Test Accuracy (%)', fontsize=11)
axes[0].set_title('Test Accuracy by Experiment', fontsize=12, fontweight='bold')
for bar, val in zip(bars1, test_acc):
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9)
axes[0].grid(True, axis='x', alpha=0.3)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

bars2 = axes[1].barh(x, overfit, color=PALETTE[:len(names)], edgecolor='white', height=0.55)
axes[1].set_yticks(x)
axes[1].set_yticklabels([n.replace(' (', '\n(') for n in names], fontsize=9)
axes[1].set_xlabel('Train Acc − Val Acc (%)', fontsize=11)
axes[1].set_title('Overfitting Gap (Lower = Better)', fontsize=12, fontweight='bold')
for bar, val in zip(bars2, overfit):
    axes[1].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].grid(True, axis='x', alpha=0.3)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Experiment Summary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Analysis & Discussion

### 10.1 Overfitting in the Baseline

The baseline model (no regularization, Adam) achieves high training accuracy but shows a significant **train–validation gap** — a classic signature of overfitting. EuroSAT contains visually and spectrally similar classes (e.g., AnnualCrop vs PermanentCrop, Highway vs River), which tempts the network to memorize subtle training-set-specific textures rather than learning robust features. The validation loss typically starts increasing after ~10 epochs while training loss continues to decrease.

### 10.2 Effect of L2 Regularization

L2 regularization (weight decay = 1e-4) penalizes large weights, shrinking them toward zero. This **smoothes the loss landscape**, making the optimizer less likely to find sharp minima that don't generalize. The train-val gap narrows noticeably, and the validation loss curve remains flatter through training. The trade-off is a slightly lower peak training accuracy because the model cannot freely fit noisy spectral patterns unique to training samples.

### 10.3 Effect of L1 Regularization

L1 regularization produces **sparse weight vectors** — many weights are driven exactly to zero. This is conceptually a form of automatic feature selection. In our CNN, L1 provides slightly less test accuracy than L2 because convolutional feature detectors benefit more from distributed small weights than sparse ones. For EuroSAT, where all 10 classes are perfectly balanced (2,700 images each), L2 provides more stable and uniform training across all classes.

### 10.4 Effect of Dropout

Dropout (p=0.4) applied to the fully-connected layers forces the network to learn **redundant representations** across multiple pathways. This is equivalent to training an ensemble of ~$2^{n}$ different sub-networks. The overfitting gap reduces substantially: the model cannot rely on a fixed set of neurons to memorize training-set patterns. One observable side effect is that reported training accuracy is lower than the true capacity (because neurons are randomly dropped during forward passes counted in training evaluation).

### 10.5 Batch Normalization Combined with Dropout

Adding BatchNorm to the Dropout model provides the best results. BatchNorm:
- Stabilizes the gradient flow through the deeper network layers
- Acts as a mild regularizer by introducing stochasticity (different batch statistics per mini-batch)
- Enables faster convergence (steeper early loss drop), particularly beneficial since we use a compact 2,000-sample training subset

The combined Dropout + BatchNorm model achieves the lowest overfitting gap and highest test accuracy.

### 10.6 SGD vs Adam

| Property | SGD (+ Momentum) | Adam |
|---|---|---|
| Convergence Speed | Slower initially | Fast convergence |
| LR Sensitivity | High (requires careful tuning) | Low (robust to LR choice) |
| Final Generalization | Often slightly better | Slightly lower in some tasks |
| Memory Cost | Low | 2× (stores $m_t$, $v_t$) |

In our experiment, Adam converges faster in the first 10 epochs. SGD with momentum (lr=0.05) + CosineAnnealing initially lags but often **catches up** in later epochs. For EuroSAT, Adam's adaptive learning rates are particularly effective because the 10 classes have equal representation, removing any optimizer-specific advantage from class imbalance handling.

### 10.7 Trade-offs Observed

- **More regularization → lower overfitting, but slower convergence and potentially lower peak accuracy**
- **Dropout alone may slow learning** because effective capacity is reduced; pairing with BN compensates
- **Adam is easier to tune** (good out-of-the-box), while **SGD requires more careful LR choice** but can generalize better
- **L1 vs L2**: L1 gives sparse, interpretable weights; L2 gives small, distributed weights — EuroSAT's spectrally rich features benefit more from distributed L2 weights


---
## 11. Conclusion

- **Overfitting is real and measurable**: The baseline model showed a significant train-validation accuracy gap on EuroSAT's spectrally similar classes. Regularization techniques systematically reduced this gap.
- **Dropout + Batch Normalization is the strongest combination**: Dropout prevents co-adaptation of neurons while BatchNorm stabilizes gradients through the deeper network layers, together yielding the best test accuracy and smallest overfitting gap.
- **L2 regularization is more suitable than L1 for CNNs on balanced datasets**: L2 (weight decay) smoothed weights without inducing sparsity, making it more compatible with distributed convolutional feature learning. EuroSAT's equal class balance (200 training images per class in our subset) makes L2 particularly stable.
- **Adam converges faster but SGD can match its generalization**: Adam is pragmatically useful when training time or tuning budget is limited, but SGD with momentum and a scheduled LR is competitive or superior for final generalization on satellite image classification.
- **Regularization is not free**: Every technique introduces a training cost — slower convergence, reduced training accuracy, or additional hyperparameters. The optimal configuration depends on dataset size, model capacity, and available compute.
